In [ ]:
%pip install -q "transformers==4.46.3" datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models #ResNet comes along with torchvision
import torchvision.transforms as T
from transformers import BertModel, BertTokenizerFast
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

/home/mitochondria/Desktop/Learning/vqa/.vqavenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [3]:
#getting the data

full = load_dataset('merve/vqav2-small', split='validation')
print('num examples:', len(full))
print('columns:', full.column_names)

ex = full[0]
print('question:', ex['question'])
print('mutiple_choice_answer:', ex['multiple_choice_answer'])
print('image:', ex['image'].size, ex['image'].mode)


Generating validation split: 100%|██████████| 21435/21435 [00:33<00:00, 631.11 examples/s] 


num examples: 21435
columns: ['multiple_choice_answer', 'question', 'image']
question: Where are the kids riding?
mutiple_choice_answer: carnival ride
image: (640, 424) RGB


In [4]:
#building vocabulary

all_answers = full['multiple_choice_answer'] #all answers from the dataset
print(f"all_answers:\n{all_answers}")
answer_counts = Counter(all_answers)  #gives {answer: its frequency}

NUM_ANSWERS = 1000 #keeping only these (minimum for 100% coverage)

top_answers = [a for a, c in answer_counts.most_common(NUM_ANSWERS)]
NUM_ANSWERS = len(top_answers)   # might be fewer than 3175

answer2idx = {a: i for i, a in enumerate(top_answers)} #answer:index
idx2answer = {i: a for i, a in enumerate(top_answers)}

covered = sum(answer_counts[a] for a in top_answers)
print('unique answers:', len(answer_counts)) #length of keys
print('kept:', NUM_ANSWERS)
print('coverage:', f'{covered / len(full):.1%}')
print('top 10:', top_answers[:10])

all_answers:
Column(['carnival ride', 'yes', 'wetsuit', '4', 'soccer', ...])
unique answers: 3181
kept: 1000
coverage: 89.5%
top 10: ['yes', 'no', '2', '1', 'white', '3', 'blue', '0', 'black', 'red']


In [5]:
answer_set = set(top_answers) #checking is faster in set compared to list using in operator

#dropping rows whose answer we did not keep
usable = full.filter(lambda a: a in answer_set, input_columns=['multiple_choice_answer']) #Huggingface dataset object.filter()
print('usable:', len(usable))

#train val test test split
split1 = usable.train_test_split(test_size=0.2, seed=42)
split2 = split1['test'].train_test_split(test_size=0.5, seed=42)
train_ds = split1['train']
val_ds = split2['train']
test_ds = split2['test']
print('train', len(train_ds), 'val', len(val_ds), 'test', len(test_ds))

Filter: 100%|██████████| 21435/21435 [00:00<00:00, 281754.07 examples/s]

usable: 19176
train 15340 val 1918 test 1918


In [9]:
#image tranforms
#we will use Resnet50
image_transform=T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(), #to 0-1
    T.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]), #numbers are the ImageNet stats ResNet was originally trained with (ResNet expects its input preshifted the exact way or accuracy drops)
])

#question tokenizer (breaks down texts and maps them to token Ids)
tokenizer=BertTokenizerFast.from_pretrained('bert-base-uncased') #using BERT text model, reads question and produces a vector
MAX_LEN=20 #since questions are short, caps the question texts at 20 tokens


In [10]:

class VQADataset(Dataset): #creating a child class using Pytorch's Dataset parent class
  def __init__(self, ds):
    self.ds=ds
  def __len__(self):
    return len(self.ds)

  def __getitem__(self,i):
    #retun one item:(image tensor, question, answer index)
    row=self.ds[i]
    image=image_transform(row['image'].convert('RGB'))
    question=row['question']
    label=answer2idx[row['multiple_choice_answer']]
    return image,question,label

def collate_fn(batch): #batch is list of (img,q,lbl)
  #called to join a list of examples into one batch
  images, questions, labels=zip(*batch) #regroups into all-images, all-questions, all-labels

  images=torch.stack(images) #piles image tensors into one big tensor with a batch dimension
  tokens=tokenizer(list(questions),padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt') #tokenizes all questions at once, padding pads shorts texts so they're of equal length, pt=pytorch tensors
  labels=torch.tensor(labels) #turn labels list into tensor
  return images, tokens, labels



BATCH_SIZE=32
#we shuffle training data only, val and test set stay in order
train_loader=DataLoader(VQADataset(train_ds), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader=DataLoader(VQADataset(val_ds), batch_size=BATCH_SIZE, num_workers=2, collate_fn=collate_fn)
test_loader=DataLoader(VQADataset(test_ds), batch_size=BATCH_SIZE, num_workers=2, collate_fn=collate_fn)


# #checking with one batch
# images,tokens,labels=next(iter(train_loader))
# print('images:',images.shape)
# print('input_ids:',tokens['input_ids'].shape)
# print('labels:',labels.shape)



In [8]:
#image encoder (ResNet 50)

#downloading the model
resnet=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1) #pretrained image model

#keep the feature extractor and drop the last classification layer
image_encoder=nn.Sequential(*list(resnet.children())[:-1])

for p in image_encoder.parameters():
  p.requires_grad=False #we skip training weights (freeze weights). we only let classifier weights learn.
image_encoder=image_encoder.eval().to(device) #set to eval mode and move to GPU


# #checking
# with torch.no_grad():
#     feats = image_encoder(images.to(device)).flatten(1) # flatten [B,2048,1,1] -> [B,2048]
# print('image features:', feats.shape)   #[B, 2048]


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/mitochondria/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:20<00:00, 5.05MB/s]


In [11]:
#question encoder (BERT)
question_encoder=BertModel.from_pretrained('bert-base-uncased') #pretrained text model
for p in question_encoder.parameters():
  p.requires_grad=False
question_encoder=question_encoder.eval().to(device)

# with torch.no_grad():
#     tokens_gpu = {k: v.to(device) for k, v in tokens.items()}
#     q_vec = question_encoder(**tokens_gpu).pooler_output
# print('question features:', q_vec.shape)

In [12]:
#fusion

class VQAModel(nn.Module):
  def __init__(self, image_encoder, question_encoder, num_answers):
    super().__init__()
    self.image_encoder=image_encoder #from ResNet
    self.question_encoder=question_encoder #from BERT

    self.classifier=nn.Sequential(
        nn.Linear(2048+768, 512),
        nn.ReLU(), #non-linearity to learn complex patterns
        nn.Linear(512,num_answers)
    )

  def forward(self,images,tokens):
    img_vec=self.image_encoder(images).flatten(1)
    q_vec=self.question_encoder(**tokens).pooler_output
    fused=torch.cat([img_vec,q_vec],dim=1) #joining them
    return self.classifier(fused)


#building the model
model=VQAModel(image_encoder, question_encoder, NUM_ANSWERS).to(device)

# out = model(images.to(device), {k: v.to(device) for k, v in tokens.items()})
# print('output:', out.shape)

In [13]:
#evaluate function

def evaluate(loader):
  model.eval()
  correct=0
  total=0
  with torch.no_grad():
    for image, tokens, labels in loader:
      image=image.to(device)
      tokens={k:v.to(device) for k,v in tokens.items()}
      labels=labels.to(device)
      preds=model(image,tokens).argmax(dim=1)
      correct+=(preds==labels).sum().item()
      total+=len(labels)
  return correct/total

In [14]:
#training loop
optimizer=torch.optim.Adam(model.classifier.parameters(), lr=1e-4)
loss_fn=nn.CrossEntropyLoss()

NUM_EPOCHS=3
for epoch in range(NUM_EPOCHS):
  model.train()
  image_encoder.eval()
  question_encoder.eval()

  running_loss=0.0
  correct=0
  total=0

  for images,tokens,labels in tqdm(train_loader,desc=f"epoch {epoch+1}"):
    images=images.to(device)
    tokens={k:v.to(device) for k,v in tokens.items()}
    labels=labels.to(device)

    optimizer.zero_grad() #clears leftover gradients from the previous step (they acculmulate by default)
    preds=model(images,tokens) #forward pass
    loss=loss_fn(preds,labels) #compute loss
    loss.backward() #backpropagation
    optimizer.step() #optimization
    running_loss+=loss.item() #extracts the loss and accumulate it
    correct+= (preds.argmax(1) == labels).sum().item()
    total+= labels.size(0)

  avg_loss=running_loss/len(train_loader)
  train_acc=correct/total
  print(f"epoch {epoch+1} loss {avg_loss:.4f} train acc {train_acc:.1%} val acc {evaluate(val_loader):.1%}")

epoch 1: 100%|██████████| 480/480 [02:32<00:00,  3.14it/s]


epoch 1 loss 4.2940 train acc 21.7% val acc 20.0%


epoch 2: 100%|██████████| 480/480 [02:26<00:00,  3.28it/s]


epoch 2 loss 3.9087 train acc 23.2% val acc 21.4%


epoch 3: 100%|██████████| 480/480 [02:27<00:00,  3.25it/s]


epoch 3 loss 3.6883 train acc 24.3% val acc 20.2%


In [15]:
#final test
print(f"test accuracy: {evaluate(test_loader):.1%}")


#predictions
model.eval()
images,tokens,labels=next(iter(test_loader))
tokens_gpu = {k: v.to(device) for k, v in tokens.items()}
with torch.no_grad():
    preds = model(images.to(device), tokens_gpu).argmax(1).cpu()#back to CPU for printing

for i in range(5):
    q = tokenizer.decode(tokens['input_ids'][i], skip_special_tokens=True)  #IDs to text
    print('Q:', q)
    print('   pred:', idx2answer[preds[i].item()], '| true:', idx2answer[labels[i].item()])

test accuracy: 22.3%
Q: how many eyes are in the picture?
   pred: yes | true: 1
Q: is she wearing a suntop?
   pred: yes | true: yes
Q: is this person wearing a tie?
   pred: yes | true: no
Q: was this photo taken at someone ' s house?
   pred: yes | true: no
Q: what kind of laptops are these?
   pred: yes | true: apple
